# 小雅免费 GPU 真实音色验收 · Kaggle T4
本 Notebook 需要 Kaggle GPU T4×2、Internet=On；**一张 T4 16GB，不能叠加显存**。用两套 Qwen3-TTS 0.6B 模型依次生成自然女声参考 WAV 和克隆试听 WAV，并输出显存与耗时报告。无需 DeepSeek API Key。注意：测试 TTS 不等于 DUIX 视频生成成功。

In [ ]:
import pathlib, json, subprocess, time, sys
out=pathlib.Path("/kaggle/working/xiaoya_results") if pathlib.Path("/kaggle/working").exists() else pathlib.Path("xiaoya_results")
out.mkdir(parents=True,exist_ok=True)
try:
    gpu=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.used","--format=csv,noheader,nounits"],capture_output=True,text=True,timeout=10,check=True)
except Exception as exc:
    raise RuntimeError("No GPU: Kaggle Settings -> Accelerator -> GPU T4 x2") from exc
print(gpu.stdout)
print("Python",sys.version.split()[0])


In [ ]:
%pip install -q --upgrade qwen-tts soundfile scipy huggingface_hub

In [ ]:
import gc, numpy as np, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel
from scipy.signal import resample_poly
from math import gcd
assert torch.cuda.is_available(), "CUDA unavailable"
assert torch.cuda.get_device_properties(0).total_memory >= 12*1024**3, "GPU0 must have at least 12GB"
torch.cuda.set_device(0)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)
ref_text="你好呀，我是小雅。今天终于见到你啦！想聊什么都可以告诉我，我会认真听着呢。"
demo_text="今天的天气真不错。我已经准备好了，可以陪你聊聊天，或者一起想想新的故事。"
style="一位二十多岁的成年女性，嗓音明亮清甜、轻盈偏高，俏皮可爱，略带撒娇感，普通话清楚自然。轻快温柔，不要刻意尖叫、哭腔或气声。"
report={"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__,"stages":[]}
def save_audio(name,wav,sr):
    a=np.asarray(wav,dtype=np.float32).reshape(-1)
    if a.size==0 or not np.isfinite(a).all() or np.max(np.abs(a))<0.005: raise RuntimeError("invalid audio")
    if int(sr)!=16000:
        d=gcd(int(sr),16000); a=resample_poly(a,16000//d,int(sr)//d)
    dur=len(a)/16000
    if not 2<=dur<=45: raise RuntimeError(f"unexpected audio duration {dur}")
    path=out/name
    sf.write(path,np.clip(a,-0.98,0.98),16000,subtype="PCM_16")
    loaded,rate=sf.read(path)
    assert rate==16000 and loaded.ndim==1
    return {"seconds":round(dur,2),"file":name,"bytes":path.stat().st_size}
def record_stage(label,t,stats):
    report["stages"].append({"name":label,"elapsed_sec":round(time.monotonic()-t,2),
        "cuda_peak_allocated_mb":round(torch.cuda.max_memory_allocated(0)/1048576,1),
        "cuda_peak_reserved_mb":round(torch.cuda.max_memory_reserved(0)/1048576,1),**stats})
    print(report["stages"][-1])


In [ ]:
# A: 真实生成小雅自然参考音
t=time.monotonic()
model=Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
    device_map="cuda:0",dtype=torch.float16,attn_implementation="eager")
wav,sr=model.generate_custom_voice(text=ref_text,language="Chinese",speaker="Vivian",instruct=style,max_new_tokens=1536)
record_stage("custom_reference",t,save_audio("ref_audio.wav",wav[0],sr))
(out/"ref_text.txt").write_text(ref_text+"\n",encoding="utf-8")
del model,wav
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)
print("Reference WAV produced; can stop now if quota low.")


In [ ]:
# B: 用 Base 模型克隆刚刚生成的声音
t=time.monotonic()
model=Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    device_map="cuda:0",dtype=torch.float16,attn_implementation="eager")
wav,sr=model.generate_voice_clone(text=demo_text,language="Chinese",
    ref_audio=str(out/"ref_audio.wav"),ref_text=ref_text,
    x_vector_only_mode=False,max_new_tokens=1536)
record_stage("base_voice_clone",t,save_audio("clone_test.wav",wav[0],sr))
del model,wav
gc.collect();torch.cuda.empty_cache()


In [ ]:
import zipfile,hashlib
report["status"]="gpu_inference_completed"
report["utc_completed"]=time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())
report["sha256"]={n:hashlib.sha256((out/n).read_bytes()).hexdigest() for n in ["ref_audio.wav","ref_text.txt","clone_test.wav"]}
(out/"gpu_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
with zipfile.ZipFile(out/"xiaoya_gpu_results.zip","w",zipfile.ZIP_DEFLATED) as z:
    for n in ["ref_audio.wav","ref_text.txt","clone_test.wav","gpu_report.json"]:z.write(out/n,n)
print("DOWNLOAD:",out/"xiaoya_gpu_results.zip")
from IPython.display import display, Audio
display(Audio(filename=str(out/"ref_audio.wav")))
display(Audio(filename=str(out/"clone_test.wav")))
